# Lab 02 - A Foundry agent grounded with Web IQ (C#)

**Optional legacy C# notebook.** Polyglot Notebooks and .NET Interactive are deprecated and no longer receive bug or security fixes. Use only if your organization permits this tooling. See [the warning and setup guide](README.md). The supported .NET 10 option is [the console app](../c-sharp-console-apps/02-foundry-web-iq-agent).

This version follows Python notebook 02 using `HttpClient` and Azure CLI authentication instead of the Python SDKs. No extra NuGet packages or Python installation are required. The small shared helper contains MCP framing, Azure authentication and request handling; the agent's tool configuration remains visible in section 3.

## Required permissions

- **Foundry Project Manager on the Foundry project:** create the MCP connection and build/run the agent with an existing model deployment. Foundry Owner on the parent resource also covers this; Foundry User alone lacks connection-write permission.
- **Web IQ profile API key:** authenticate calls to the Web IQ MCP server. No subscription Owner or tenant administrator role is required.

**Resource prefix:** set `WORKSHOP_PREFIX` in `.env`, for example `alice01` (3-20 lowercase letters/digits, starting with a letter). This lab creates or updates `alice01-webiq-mcp` and reuses a matching Web IQ agent, or creates `alice01-webiq-agent` if none exists. Reuse the prefix in notebook 03. Prefixes prevent naming collisions, not enforce access isolation; privileged project members can still access connections.

**Goal:** connect a Foundry prompt agent to the Web IQ MCP server and inspect how it retrieves evidence. Allow 35 minutes.

Architecture: **notebook -> Entra ID -> Foundry agent + deployed model -> authenticated MCP connection -> Web IQ**. This is explicitly Web IQ MCP (`api.microsoft.ai/v3/mcp`), **not** a substituted Bing grounding or built-in web-search tool.

**Before running:** have a deployed Foundry project and chat model, set `WEBIQ_API_KEY` and `WORKSHOP_PREFIX` in `.env`, run the kernel from the `c-sharp-notebooks` directory, and run `az login` in the deployment tenant. Settings load from `..\.env`, then `..\.env.azure`, just like notebook 01. File values override existing environment variables; `.env.azure` wins if both files define the same setting. Missing values raise an error instead of prompting. Calls incur both model-token and Web IQ usage charges.

The deployment defaults to East US (`eastus`); Web IQ remains global. Existing deployments keep their current region until you redeploy. After changing deployment regions, restart the kernel to reload `.env.azure` and recreate the connection and agent in the new Foundry project. Search and Blob Storage are provisioned for later modules and are not used by this notebook.

**C# setup:** select the .NET Interactive C# kernel and start it from `c-sharp-notebooks`. Use a runtime compatible with your installed legacy kernel; the .NET 10 SDK alone does not guarantee compatibility. The setup cell imports the same simple settings loader as C# lab 01 and `FoundryWorkshop.cs` from the console app. The loader supports quoted `KEY=VALUE` settings and comments, but not dotenv interpolation or multiline values. It never prints the key.

## Foundry control plane (reference)

<img src="../docs/foundry-control-plane.png" alt="Foundry resource hierarchy showing tenant, subscription, resource groups, Foundry accounts, account-level model deployments, and project-level connections" width="700">

Model deployments belong to the Foundry account; connections belong to a project. This lab reuses an existing model deployment and creates its Web IQ MCP connection inside the project. These resource boundaries help explain the role assignment scopes in the next reference.

Creating the connection is a management-plane operation through Azure Resource Manager. Creating and running this prompt agent use the Foundry data plane; this diagram shows resource organization, not the runtime request flow.

## Foundry built-in roles (reference)

![Six Foundry built-in roles compared across resource management, project connections, model deployment, runtime APIs, agent publishing and restricted role delegation, with scope requirements](../docs/foundry-built-in-roles.png)

Use this chart as an overview, not a substitute for the **Required permissions** above. Role assignment scope matters, and general project-development access does not necessarily include connection-write permission.

**Workshop access:** `grant-workshop-access.ps1` assigns listed attendees **Foundry Owner on the shared Foundry account**, inherited by its projects. This covers this lab's requirements; **Foundry Project Manager on the project** is a narrower alternative for lab 02 alone. The workshop grant is not subscription Owner or tenant administrator.

**Foundry Account Owner and Foundry Owner are different roles.** Account Owner does not grant project data actions. Foundry Agent Consumer authorizes published endpoint access; Foundry Project Runtime User authorizes the project Responses API.

**Project creation:** the chart follows the explicit `Microsoft.CognitiveServices/accounts/projects/*` permission in the Foundry Project Manager role definition, which requires assignment at the parent account to create projects. Microsoft Learn's simplified Foundry overview table currently differs on this entry. A project-only assignment does not grant creation of sibling projects.

In [ ]:
#r "System.Net.Http.Json"
#r "System.Text.Json"
#!import ..\c-sharp-console-apps\01-web-iq-apis\Workshop.cs
#!import ..\c-sharp-console-apps\02-foundry-web-iq-agent\FoundryWorkshop.cs

using System;
using System.Net.Http;
using System.Text.Json.Nodes;
using System.Threading.Tasks;

FoundrySettings settings = FoundrySettings.Load("..");
HttpClient client = new HttpClient { Timeout = TimeSpan.FromSeconds(120) };
FoundryWorkshop lab = new FoundryWorkshop(client, settings);
Console.WriteLine("Foundry project: " + settings.ProjectEndpoint);
Console.WriteLine("MCP Endpoint: " + FoundryWorkshop.McpUrl);

## 1. Discover what this account can actually do

The Web IQ MCP server exposes `web`, `news`, `images`, `videos` and `browse`, let's look at tool discovery

Discovery HTTP errors, including `401` and `429`, and connection/timeouts print diagnostics without a traceback. A failed discovery clears the previous tool list, and the resource-creation cells skip. For `401`, verify that `WEBIQ_API_KEY` is a Web IQ profile key, not an Azure resource key. Fix the error and rerun setup/discovery before continuing.

### Explore with MCP Inspector (optional)

[MCP Inspector](https://github.com/modelcontextprotocol/inspector) lets you discover tools, inspect their schemas and test calls in a browser. Install a supported Node.js version (see the Inspector's [requirements](https://github.com/modelcontextprotocol/inspector#quick-start-development)), then run this in a **terminal**, not a Python notebook cell:

```powershell
npx @modelcontextprotocol/inspector
```

`npx` downloads and runs the package without a global installation. Leave the terminal running while using Inspector.

<img src="../docs/mcp-inspector-startup.png" alt="MCP Inspector terminal startup with the local access token redacted" width="700">

Open the **MCP Inspector Web** URL printed in your own terminal, not the Sandbox URL. It includes a local Inspector access token; the screenshot above is redacted, so do not copy its example URL or share your token.

On the **Servers** tab, click **Add Servers** and add a remote HTTP server with these settings:

<img src="../docs/mcp-inspector-add-server.png" alt="MCP Inspector Servers tab with Add Servers highlighted" width="900">

| Setting | Value |
|---|---|
| Server name | `Web IQ` (a friendly name of your choice) |
| Transport | **Streamable HTTP** |
| MCP endpoint / URL | `https://api.microsoft.ai/v3/mcp` |
| Request header name | `x-apikey` |
| Request header value | Paste the value of `WEBIQ_API_KEY` from the repository-root `.env` file. |

Inspector does **not** automatically read this workshop's `.env` file. Paste the key value itself, not the variable name, quotes or `WEBIQ_API_KEY=` prefix. This is your **Web IQ profile key**, not an Azure resource key or the local Inspector access token. Keep it out of notebook cells, screenshots, exported configurations and Git.

Save/connect the server, then open **Tools** to inspect the available tool schemas and try a call. Inspector shows the tools exposed by your profile; the Foundry agent's `allowed_tools` setting does not restrict this separate client. Actual tool calls incur usage charges, so use public, non-sensitive queries.

### Discover tools from C#

The cell below connects directly to `https://api.microsoft.ai/v3/mcp` over **Streamable HTTP**, authenticating with your Web IQ profile key in the `x-apikey` header.

The shared `DiscoverToolsAsync()` helper sends **`initialize`**, then **`notifications/initialized`** to establish the MCP session. It then sends **`tools/list`** requests, following pagination to discover the available tools and their schemas. It accepts JSON or Server-Sent Events responses and forwards the negotiated protocol version and any MCP session ID.

These are **MCP JSON-RPC requests over HTTP**, rather than conventional REST calls. This cell discovers tools; it does **not** execute a search or invoke the Foundry agent.

In [ ]:
JsonArray availableTools = null;
availableTools = await lab.DiscoverToolsAsync();


## 2. Store the workshop API key in your own Foundry project connection

This management-plane PUT creates or updates `{prefix}-webiq-mcp` as a `RemoteTool` connection with a custom `x-apikey` header. The name is derived from `WORKSHOP_PREFIX`. The key stays out of the agent definition and notebook outputs. The cell deliberately does not display the connection response because it may contain credentials.

**Role for this step:** Foundry Project Manager on this project includes `Microsoft.CognitiveServices/accounts/projects/connections/write`. Foundry User alone does not include that management-plane action. Foundry Owner on the parent resource is a sufficient but broader alternative, not a prerequisite.

A Foundry project connection protects a shared service credential, but authorized project members may access it. Use a workshop-only key, restrict project membership, and revoke the key afterwards. This authoring exercise requires connection management permission; users who only invoke a prebuilt agent do not need that permission.

The connection can also be created in the Foundry portal: add a remote MCP tool at the Web IQ MCP URL with key-based credential name `x-apikey`.

In [ ]:
bool connectionConfigured = false;
if (availableTools != null)
    connectionConfigured = await lab.ConfigureConnectionAsync();
else
    Console.WriteLine("Resolve the discovery error before creating the connection.");

### Verify the connection in the Foundry portal

Go to the Foundry Portal: [ai.azure.com](https://ai.azure.com)

Open the Foundry project, select **Manage**, then **Connections**. Find `{prefix}-webiq-mcp` and confirm:

- **Category:** `RemoteTool`
- **Auth method:** `Custom Keys`
- **Target:** `https://api.microsoft.ai/v3/mcp`

![Foundry Manage page showing the webiq-mcp connection under Connected resources](../docs/foundry-web-iq-connection.png)

Your resource names will differ. This confirms that the connection exists, not that its credentials work. The portal listing is not a credential test; the agent's successful MCP tool call later in this lab verifies end-to-end access.

## 3. Create or update the Web IQ agent

We are going to create an agent named `{prefix}-webiq-agent` that will be a prompt agent connected to the Web IQ MCP endpoint.  Each run of the cell will create a new version of the agent.

The agent can use `web` to find sources and `browse` to inspect source URLs. `require_approval="never"` lets these tools run without interactive prompts. Tool calls send their arguments to an external web service.

We could also include `images` and `videos` (or `news`) in `allowed_tools` if wanted and exposed by the profile in tool discovery. They are tools on the same Web IQ MCP endpoint, so no additional endpoint or connection is needed.

`FoundryWorkshop.Instructions` contains the same research and untrusted-page instructions as Python lab 02. The shared helper lists all agent pages, prefers the stable matching name, and refuses to replace a same-name agent that is not a Web IQ prompt agent.

In [ ]:
JsonObject agent = null;
JsonObject conversation = null;
JsonObject definition = new JsonObject
{
    ["kind"] = "prompt",
    ["model"] = settings.Model,
    ["instructions"] = FoundryWorkshop.Instructions,
    ["tools"] = new JsonArray(new JsonObject
    {
        ["type"] = "mcp",
        ["server_label"] = "webiq",
        ["server_url"] = FoundryWorkshop.McpUrl,
        ["project_connection_id"] = settings.ConnectionId,
        ["allowed_tools"] = new JsonArray("web", "browse"),
        ["require_approval"] = "never",
    }),
};
if (availableTools != null && connectionConfigured)
{
    agent = await lab.CreateAgentVersionAsync(definition);
    if (agent != null)
        conversation = await lab.CreateConversationAsync();
}
else
    Console.WriteLine("Resolve discovery/connection errors before creating the agent.");

## 4. Run a question without interactive prompts

**This section uses the Responses API** to send your question to the Foundry agent through `lab.AskAsync(...)`. The shared helper uses `HttpClient` to POST to `{projectEndpoint}/openai/v1/responses`: `input` contains the question, `conversation` identifies the conversation to retain context, and `agent_reference` is a top-level JSON field containing the agent name and version. These are the same fields sent by Python's `openai.responses.create(...)`; there is no `extra_body` wrapper on the wire.

[Microsoft Learn: Generate responses with a Foundry agent](https://learn.microsoft.com/azure/foundry/agents/concepts/runtime-components#generate-responses).

The agent can call the allowed Web IQ `web` and `browse` tools without prompting. The helper submits each question to the same conversation, retaining context for follow-up questions. HTTP errors (including `429`), connection/timeouts, and failed or incomplete executions print diagnostics without a traceback and return no answer. Verification and follow-up cells do not treat a failed request as a successful response.

For `429`, inspect the printed error and retry headers. Wait before rerunning the question; check the model's quota/rate limits if it persists. This C# HTTP example performs no automatic retries. Unlike the Python SDK, it does not add SDK-level retries; rerun the question manually after resolving the error. HTTP error bodies are limited to 8000 characters, as in notebook 01.

Rerun section 3 after changing the agent configuration: it creates a new version and starts a fresh conversation, so pending approvals from an older run do not carry over. An unexpected approval request raises an error instead of prompting.

In [ ]:
async Task<JsonObject> AskAgentAsync(string question)
{
    if (agent == null || conversation == null)
    {
        Console.WriteLine("Create the agent version and conversation in section 3 first.");
        return null;
    }
    return await lab.AskAsync(question, conversation, agent);
}

In [ ]:
string question =
    "Use Web IQ to find recent Microsoft announcements about AI agents. "
    + "Give three short evidence-backed findings, with a source URL for each "
    + "and one explicit uncertainty. Do not assume a crawl date is a publication date.";
JsonObject response = null;
response = await AskAgentAsync(question);

## 5. Verify retrieval, not just fluent prose

A coherent answer is not proof of grounding. Inspect the actual MCP calls and errors. The next cell requires at least one `webiq` call with a non-empty result and no reported error. Then manually open the cited sources and check whether each claim is supported.

This is an integration check, **not** a claim that every citation is factually correct. Source URLs written by the model still need review; formal URL-citation annotations may not be present for every MCP-based answer.

In [ ]:
bool grounded = lab.VerifyRetrieval(response);

## 6. Challenge: compare direct retrieval and agent orchestration

Reuse your lab 01 briefing topic. Ask a follow-up that requires the agent to acknowledge uncertainty. Compare the returned sources, number of API calls, control over parameters, latency and the quality of citations.

The next cell always asks a follow-up in the same conversation after a successful first response. This is another billable agent turn. If the first request failed, resolve that error before continuing.

For an extension, add `news`, `images` or `videos` to `allowed_tools` only if they appeared in discovery. Update the instructions, create a new agent version, update `agent_reference`, and ask a question that actually requires the new tool. A configured tool that is never called does not demonstrate its capability.

**Acceptance criteria:** at least one confirmed successful Web IQ MCP call, three claims with supporting URLs, one acknowledged uncertainty, and a clear explanation of when you would prefer the direct REST approach.

In [ ]:
JsonObject followUp = null;
if (response != null)
    followUp = await AskAgentAsync(
        "Search again for primary sources. Which earlier claim has the weakest evidence, and why?");
else
    Console.WriteLine("No initial agent response. Resolve the request error before asking a follow-up.");

## 7. Finish

The agent, its versions, conversation and MCP connection are retained. Notebook 03 reuses the connection; rerunning this notebook updates the matching agent rather than creating another agent name. The discovery helper already closes its MCP session when present. The next cell only closes the local HTTP client. Clear notebook outputs before sharing.

In [ ]:
client.Dispose();
Console.WriteLine("Local HTTP client closed. Azure resources and the conversation are retained.");

## References

- [Web IQ MCP tools and account-scoped discovery](https://learn.microsoft.com/web-iq/integrations/mcp)
- [Foundry MCP integration and approvals](https://learn.microsoft.com/azure/foundry/agents/how-to/tools/model-context-protocol)
- [Foundry MCP authentication and shared-key connections](https://learn.microsoft.com/azure/foundry/agents/how-to/mcp-authentication)
- [Web IQ pricing](https://learn.microsoft.com/web-iq/resources/pricing)